# 06 — Ten-stage recognition evaluation

This notebook launches the fixed ten-stage recognition evaluator for exported Guesser serving bundles. It scores validation by default; the held-out test split requires a frozen record and must not be used for tuning. This is recognition evaluation, not training, gameplay scoring, or hint-assisted ranking.

The evaluator lives in `models/guesser/recognition/` and runs in the Python 3.13 model project through `uv run`. Do not import its modules into this Python 3.12 notebook kernel.


## Protocol and defaults

Each validation or test original is rendered at ten fixed prefixes: 10%, 20%, …, 100% of its recorded drawing time. Both A/B bundles receive the same preprocessed batch and the same input hash, so the comparison is paired. The end-to-end stage curve counts blank, uncertain and unchanged views as unsuccessful; the policy table treats them as not emitted and reports coverage, accuracy among emitted guesses and overall correctness separately.

By default the runner scores a fixed balanced subset of 50 originals per class: 17,250 originals and 172,500 views per arm. `FULL = True` uses all 172,500 validation originals and 1,725,000 views per arm. It saves full float32 logits so temperature calibration is exact. One scalar temperature per arm is fitted on validation views, and the 0–1 policy thresholds are reported as a trade-off rather than a single game decision.

The A/B comparison uses a stratified bootstrap over whole drawings within class, keeping all ten stages together. The provisional selection goal is B gaining at least +5 mean early macro top-1 points with no more than 2 complete top-1 points lost.


## Launch and development workflow

From the repository root, sync the model environment, then launch JupyterLab through the evaluation environment with the repository as its contents root:

```sh
uv sync --project models/guesser --frozen
uv run --project models/guesser/evaluation jupyter lab --ServerApp.root_dir=.
```

The model scripts are the source of truth. The notebook invokes `recognition.evaluate` as a checked subprocess from `models/guesser/`; it does not copy the runner into the notebook kernel. The subprocess environment removes inherited `PYTHONPATH` and `PYTHONHOME` so unrelated Python installations cannot shadow the model environment.

> **Expected cross-project warning:** `uv` may report that the evaluation `VIRTUAL_ENV` does not match `models/guesser/.venv`. This is intentional: the notebook kernel uses Python 3.12, while the evaluator uses the separate Python 3.13 model environment. `--project` deliberately selects the model environment; do not add `--active`, which would force the notebook environment. This exact mismatch warning is harmless by itself; other errors or nonzero command results still indicate a problem.


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
from guesser_evaluation import notebook_tools

NOTEBOOK_TOOLS = notebook_tools.module_source_path("notebook_tools.py")
REPOSITORY_ROOT = NOTEBOOK_TOOLS.parents[5]
MODEL_PROJECT = REPOSITORY_ROOT / "models" / "guesser"
if not (MODEL_PROJECT / "recognition" / "evaluate.py").is_file():
    raise FileNotFoundError(f"Ten-stage evaluator not found under {MODEL_PROJECT}")

print("Repository:", REPOSITORY_ROOT)
print("Model project:", MODEL_PROJECT)


## Validation or test?

The runner scores exactly one split per invocation, chosen by `SPLIT`. The two are not interchangeable.

| | `SPLIT = "validation"` | `SPLIT = "test"` |
|---|---|---|
| Data | `manifests/validation.parquet` | `manifests/test.parquet` |
| Role | tuning split | held-out split |
| Temperature | fitted on this run's views | read from the freeze record |
| Thresholds | every 0.00-0.95 reported as a trade-off | one frozen value per arm |
| Requires | nothing | `FREEZE_RECORD` and `VALIDATION_REPORT` |
| Output root | `evaluation/ten-stage/validation/` | `evaluation/ten-stage/test/` |

Validation is where two decisions get made: it fits the temperature, and its policy table is what you choose an operating threshold from. Because those decisions are made on it, its own numbers are not an independent result. They are the evidence you tune with.

The test split never influenced training, checkpoint selection, temperature or threshold. It is the only split whose numbers may be reported as the model's recognition performance. Its guard is deliberately strict: the runner refuses to fit a temperature or sweep thresholds there, and refuses to start without a freeze record whose pinned hash matches the validation report on disk.

So the order is fixed, and each step is enforced by code rather than convention:

1. `SPLIT = "validation"` - run it, then read that split's `report.md`.
2. Choose one threshold per arm from the policy table.
3. Run the freeze cell with `WRITE_FREEZE_RECORD = True`.
4. `SPLIT = "test"` with `FREEZE_RECORD` and `VALIDATION_REPORT` set - run it.

Each split writes to its own output root, so step 4 cannot overwrite the validation report that step 3 pins by hash.


## Options

The defaults target the completed production A/B bundles. `SPLIT = "validation"` is the only tuning split. Each split writes to its own output root, so the held-out test run never overwrites the validation report its freeze record pins. Leave `SAVE_LOGITS = True` for exact temperature calibration and for any later frozen test run. `RESUME = True` is safe for a fresh output root: when `subset.json` is absent, the runner starts a new run; when a valid run exists, it skips class shards whose hashes match `progress.json`. `OVERWRITE = True` replaces the output directory and cannot be combined with resume.

`FREEZE_RECORD` and `VALIDATION_REPORT` are only set for a held-out test run. The freeze record must pin the bundle hashes, temperature and threshold from a completed validation report.

`WORKERS` sets the render worker processes. `0` renders in the launching process; a worker count near the physical core count is the practical ceiling. `BATCH_SIZE` counts views per model forward and also caps how much each worker holds in memory.

On a full run this is a long job, so budget hours rather than minutes and leave the machine otherwise idle. Each original is rendered at ten prefixes on the CPU and both arms are scored on every usable view, so cost grows with the split and the preprocessing candidate, not only with the model. `WORKERS` renders in parallel and overlaps that work with inference; raising it past the physical core count stops helping and costs memory, since every worker holds one batch of views. The per-class lines show how far the run has reached, and `RESUME = True` continues from the last finished class after an interruption.


In [ ]:
from pathlib import Path

# Execution is opt-in; review the options, paths and resource cost.
RUN_EVALUATION = True

# What to score. Both splits hold the same number of originals; only their
# role differs, so read the protocol section above before changing SPLIT.
DATA_PROFILE = "production"  # 'small', 'medium', or 'production'
SPLIT = "test"  # 'validation' = tuning split, 'test' = held-out split
MODEL_IDS = ("guesser-vit-a-v1", "guesser-vit-b-v1")  # one bundle or an A/B pair

# How much of the split to score. FULL covers every original and is the only
# setting worth reporting; a per-class limit exists for smoke checks, and its
# numbers must not be presented as results.
LIMIT_PER_CLASS = None
FULL = True

# Cost and memory. WORKERS renders and preprocesses in parallel worker
# processes; BATCH_SIZE is the number of usable views in one model forward and
# also bounds how many views each worker holds.
BATCH_SIZE = 128
WORKERS = 10

# Scoring. The temperature is fitted on at most CALIBRATION_SAMPLES usable
# validation views. SAVE_LOGITS keeps full float32 logits, which the fit needs
# and which a later frozen test run also requires.
BOOTSTRAP = 1000
CALIBRATION_SAMPLES = 50000
SAVE_LOGITS = True

# Existing artifacts. RESUME continues a partial run and starts cleanly on an
# empty root; OVERWRITE deletes an existing run. They cannot be combined.
OVERWRITE = False
RESUME = True

DEVICE = "cuda"  # 'auto', 'cpu', or 'cuda'

# Test only; both must stay None for a validation run. The freeze record was
# written once the validation run finished and pins that run's report hash, so do
# not re-run the validation launch cell afterwards: it regenerates report.md with
# new timestamps, which changes the hash and makes this run refuse to start.
# Derive both paths so they follow DATA_PROFILE instead of being typed by hand.
FREEZE_ROOT = MODEL_PROJECT / "checkpoints" / f"notebook-{DATA_PROFILE}-full" / "evaluation"
FREEZE_RECORD = FREEZE_ROOT / "ten-stage-freeze.json"
VALIDATION_REPORT = FREEZE_ROOT / "ten-stage" / "validation" / "report.md"

# Reject contradictory settings here so a bad combination fails in this cell
# rather than partway through a multi-hour subprocess.
if DATA_PROFILE not in {"small", "medium", "production"}:
    raise ValueError("Select small, medium, or production.")
if SPLIT not in {"validation", "test"}:
    raise ValueError("SPLIT must be validation or test.")
if not MODEL_IDS or len(MODEL_IDS) > 2:
    raise ValueError("MODEL_IDS must contain one or two bundle IDs.")
if LIMIT_PER_CLASS is not None and LIMIT_PER_CLASS < 1:
    raise ValueError("LIMIT_PER_CLASS must be positive or None.")
if not FULL and LIMIT_PER_CLASS is None:
    raise ValueError("Set FULL = True or LIMIT_PER_CLASS for a subset run.")
if BATCH_SIZE < 1 or BOOTSTRAP < 0 or CALIBRATION_SAMPLES < 1:
    raise ValueError("Check BATCH_SIZE, BOOTSTRAP and CALIBRATION_SAMPLES.")
if WORKERS < 0:
    raise ValueError("WORKERS must be non-negative.")
if OVERWRITE and RESUME:
    raise ValueError("OVERWRITE and RESUME are mutually exclusive.")
# A test run may not fit anything, so it needs the frozen temperature and
# threshold plus the validation report they were derived from.
if SPLIT == "test":
    if FREEZE_RECORD is None or VALIDATION_REPORT is None:
        raise ValueError("A test run requires FREEZE_RECORD and VALIDATION_REPORT.")
    if not SAVE_LOGITS:
        raise ValueError("A test run requires SAVE_LOGITS = True.")
elif FREEZE_RECORD is not None or VALIDATION_REPORT is not None:
    raise ValueError("Freeze inputs are only valid for SPLIT = 'test'.")

# Derive every path from the profile and split so a validation run and a test
# run can never share an output root.
PREPARED_DIR = MODEL_PROJECT / "data" / ".local" / "data" / DATA_PROFILE / "prepared"
CHECKPOINT_ROOT = MODEL_PROJECT / "checkpoints" / f"notebook-{DATA_PROFILE}-full"
BUNDLE_ROOT = CHECKPOINT_ROOT / "bundles"
OUTPUT_ROOT = CHECKPOINT_ROOT / "evaluation" / "ten-stage" / SPLIT
if FREEZE_RECORD is not None:
    FREEZE_RECORD = Path(FREEZE_RECORD).expanduser().resolve()
if VALIDATION_REPORT is not None:
    VALIDATION_REPORT = Path(VALIDATION_REPORT).expanduser().resolve()

# Print the resolved plan so the run can be checked before it starts.
print("Evaluation enabled:", RUN_EVALUATION)
print("Profile / split:", DATA_PROFILE, SPLIT)
print("Prepared data:", PREPARED_DIR)
print("Bundles:", BUNDLE_ROOT)
print("Output root:", OUTPUT_ROOT)
print("Originals per class:", None if FULL else LIMIT_PER_CLASS)
print("Full validation:", FULL)
print("Batch / bootstrap / calibration samples:", BATCH_SIZE, BOOTSTRAP, CALIBRATION_SAMPLES)
print("Render workers:", WORKERS)
print("Save logits:", SAVE_LOGITS)
print("Overwrite / resume:", OVERWRITE, RESUME)
if SPLIT == "test":
    print("Freeze record:", FREEZE_RECORD)
    print("Validation report:", VALIDATION_REPORT)


## Readiness check

This read-only check confirms the selected split manifest and both bundle manifests exist, verifies each manifest's model ID, and probes the model environment with `--no-sync`. It does not load the weights or start evaluation. The evaluator repeats the full bundle hash and label checks before scoring.


In [ ]:
import json
import os
import subprocess

MODEL_ENV = dict(os.environ)
MODEL_ENV.pop("PYTHONPATH", None)
MODEL_ENV.pop("PYTHONHOME", None)
MODEL_ENV["PYTHONUNBUFFERED"] = "1"

manifest_path = PREPARED_DIR / "manifests" / f"{SPLIT}.parquet"
if not manifest_path.is_file():
    raise FileNotFoundError(f"{SPLIT} manifest not found: {manifest_path}")

for model_id in MODEL_IDS:
    bundle_manifest = BUNDLE_ROOT / model_id / "manifest.json"
    if not bundle_manifest.is_file():
        raise FileNotFoundError(f"Bundle manifest not found: {bundle_manifest}")
    payload = json.loads(bundle_manifest.read_text(encoding="utf-8"))
    if payload.get("modelId") != model_id:
        raise ValueError(f"{bundle_manifest} does not describe {model_id}")

probe_code = (
    "import sys, numpy, pyarrow, torch, timm, safetensors, guesser_data; "
    'print("python=" + sys.version.split()[0]); '
    'print("executable=" + sys.executable); '
    'print("torch=" + torch.__version__); '
    'print("cuda_available=" + str(torch.cuda.is_available()))'
)
probe = subprocess.run(
    [
        "uv",
        "run",
        "--project",
        str(MODEL_PROJECT),
        "--frozen",
        "--no-sync",
        "python",
        "-c",
        probe_code,
    ],
    cwd=MODEL_PROJECT,
    env=MODEL_ENV,
    capture_output=True,
    text=True,
    check=False,
)
if probe.returncode != 0:
    raise RuntimeError("Model environment probe failed:\n" + probe.stderr)
print(probe.stdout.strip())
print("Prepared manifest:", manifest_path)
print("Bundles:", ", ".join(MODEL_IDS))
print("Output root:", OUTPUT_ROOT)


## Run the evaluator (explicit opt-in)

When enabled, the cell launches `recognition.evaluate`. The runner validates every bundle hash and label file, selects the fixed subset, streams the ten views through both arms, and writes class shards and logits before computing metrics. Interrupting the cell asks the evaluator to stop; a later `RESUME = True` run skips class shards whose hashes match `progress.json`. If no `subset.json` exists yet, `RESUME = True` starts a new run instead of failing. Do not run two evaluators against the same output root at the same time.


In [ ]:
if not RUN_EVALUATION:
    print("Evaluation disabled. Review the options, then set RUN_EVALUATION = True to launch.")
else:
    evaluation_command = [
        "uv",
        "run",
        "--project",
        str(MODEL_PROJECT),
        "--frozen",
        "--no-sync",
        "python",
        "-m",
        "recognition.evaluate",
        "--prepared-dir",
        str(PREPARED_DIR),
        "--bundle-root",
        str(BUNDLE_ROOT),
        "--output-dir",
        str(OUTPUT_ROOT),
        "--split",
        SPLIT,
        "--batch-size",
        str(BATCH_SIZE),
        "--workers",
        str(WORKERS),
        "--bootstrap",
        str(BOOTSTRAP),
        "--calibration-samples",
        str(CALIBRATION_SAMPLES),
        "--device",
        DEVICE,
    ]
    for model_id in MODEL_IDS:
        evaluation_command.extend(["--model-id", model_id])
    if FULL:
        evaluation_command.append("--full")
    elif LIMIT_PER_CLASS is not None:
        evaluation_command.extend(["--limit-per-class", str(LIMIT_PER_CLASS)])
    if not SAVE_LOGITS:
        evaluation_command.append("--no-save-logits")
    if OVERWRITE:
        evaluation_command.append("--overwrite")
    if RESUME:
        evaluation_command.append("--resume")
    if SPLIT == "test":
        evaluation_command.extend(
            [
                "--freeze-record",
                str(FREEZE_RECORD),
                "--validation-report",
                str(VALIDATION_REPORT),
            ]
        )
    print("Launching:", " ".join(evaluation_command), flush=True)
    try:
        completed = notebook_tools.run_interruptible_command(
            evaluation_command, cwd=MODEL_PROJECT, env=MODEL_ENV
        )
    except subprocess.CalledProcessError as error:
        # The runner exits 130 after a clean operator stop. Any other
        # non-zero status is a real failure and must still surface.
        if error.returncode != 130:
            raise
        completed = False

    if completed:
        print("Evaluation command completed successfully.")
    else:
        # Interrupted on purpose: every finished class shard is already on
        # disk with its hash recorded, so rerunning this cell continues from
        # the last finished class instead of starting over.
        print(
            f"{SPLIT.capitalize()} stopped. Finished classes are saved; "
            "rerun this cell with RESUME = True to continue.",
            flush=True,
        )


## Summary, stage curves and exported figures

The summary is read from `summary.json`, not from training curves. The stage table shows end-to-end top-1 and top-3 at each prefix. The summary table shows mean early macro/overall top-1, complete top-1/top-3, and blank/uncertain rates. The policy table separates coverage from accuracy among emitted guesses. The A/B section reports paired bootstrap intervals and whether the provisional goal was met.

**These displays are not the deliverable.** Notebook outputs are stripped before they reach Git, so every number and curve is also written to `<output root>/figures/`: the tables as CSV, the curves as PNG and SVG. The policy sweep appears only for a validation run, because a frozen test run reports a single threshold rather than a trade-off. The cells below print each written path and display the saved files, so what you see is what the run produced.


In [ ]:
import json

import pandas as pd
from IPython.display import display

SUMMARY_PATH = OUTPUT_ROOT / "summary.json"
if not SUMMARY_PATH.is_file():
    print("No summary yet. Run the evaluator or check OUTPUT_ROOT.")
else:
    summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
    rows = []
    for model_id, model in summary["models"].items():
        rows.append(
            {
                "model": model_id,
                "mean early top-1 (macro)": model["meanEarlyTop1Macro"],
                "mean early top-1 (overall)": model["meanEarlyTop1"],
                "complete top-1": model["completeTop1"],
                "complete top-3": model["completeTop3"],
                "blank": model["blankRate"],
                "uncertain": model["uncertainRate"],
            }
        )
    display(pd.DataFrame(rows).set_index("model"))
    print("Split:", summary["split"])
    print("Originals / views:", summary["originals"], "/", summary["views"])
    print("Calibration:", summary["calibration"])
    if summary["comparison"]:
        print("A/B comparison:", json.dumps(summary["comparison"], indent=2))


In [ ]:
from IPython.display import Image, display

from guesser_evaluation import report_figures

if not SUMMARY_PATH.is_file():
    print("No summary yet; run the evaluator first.")
else:
    # Notebook outputs are stripped on commit, so every table and curve is
    # written next to report.md instead of living only in this session.
    FIGURES_DIR = OUTPUT_ROOT / "figures"
    for path in report_figures.export_all(SUMMARY_PATH, FIGURES_DIR):
        print("wrote", path)
    # Show the files that were written, not a figure that exists only in memory.
    for name in ("stage-top1", "delta-by-stage", "coverage-by-stage"):
        image_path = FIGURES_DIR / f"{name}.png"
        if image_path.is_file():
            display(Image(filename=str(image_path)))


In [ ]:
if SUMMARY_PATH.is_file():
    summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
    for model_id, model in summary["models"].items():
        print(f"\n{model_id}")
        policy = pd.DataFrame(model["policy"])
        policy = policy[
            ["threshold", "coverage", "accuracyEmitted", "overallCorrectness"]
        ]
        policy["threshold"] = policy["threshold"].map(lambda value: f"{value:.2f}")
        display(policy)
else:
    print("No summary yet; run the evaluator first.")


## Optional freeze record for a held-out test run

A test run must not fit temperature or choose a threshold. After a full validation run, review that split's `report.md` and select one policy threshold per arm. This optional cell writes a freeze record from the validation `summary.json` and `report.md`; it is disabled by default. It records the validation report hash and the bundle identity fields already stored in the summary. Run it while `SPLIT` is still `"validation"` and after re-running the options cell, because `OUTPUT_ROOT` and the paths derived from it follow `SPLIT`. Do not edit the record after seeing test results.


In [ ]:
import hashlib
from datetime import UTC, datetime

# One operating point per arm, chosen from the validation policy table before
# any test result exists. Editing these after seeing test numbers invalidates
# the held-out claim, which is the whole reason this record exists.
WRITE_FREEZE_RECORD = False
FROZEN_THRESHOLDS = {
    "guesser-vit-a-v1": 0.5,
    "guesser-vit-b-v1": 0.5,
}
# Lives beside the split directories, not inside one: a single record pins the
# validation report and is then consumed by the test run.
FREEZE_OUTPUT = CHECKPOINT_ROOT / "evaluation" / "ten-stage-freeze.json"

if not WRITE_FREEZE_RECORD:
    print("Freeze record disabled. Set WRITE_FREEZE_RECORD = True after reviewing validation.")
elif not SUMMARY_PATH.is_file() or not (OUTPUT_ROOT / "report.md").is_file():
    raise FileNotFoundError("Run validation first; summary.json and report.md are required.")
else:
    summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
    if summary["split"] != "validation":
        raise ValueError("A freeze record must be built from a validation run.")
    if summary["calibration"]["mode"] != "temperature":
        raise ValueError("A freeze record requires a temperature-calibrated validation run.")
    for model_id, threshold in FROZEN_THRESHOLDS.items():
        if not 0 <= threshold <= 1:
            raise ValueError(f"Threshold for {model_id} must be in [0, 1].")
    report_path = OUTPUT_ROOT / "report.md"
    models = {}
    for model_id, model in summary["models"].items():
        if model_id not in FROZEN_THRESHOLDS:
            raise ValueError(f"Set FROZEN_THRESHOLDS for {model_id}.")
        identity = model["identity"]
        models[model_id] = {
            "artifactRevision": identity["artifactRevision"],
            "weightsSha256": identity["weightsSha256"],
            "labelsSha256": identity["labelsSha256"],
            "preprocessing": identity["preprocessing"],
            "preprocessingHash": identity["preprocessingHash"],
            "temperature": summary["calibration"]["temperatures"][model_id],
            "threshold": FROZEN_THRESHOLDS[model_id],
        }
    payload = {
        "schema": "guesser-ten-stage-freeze-v1",
        "createdUtc": datetime.now(UTC).isoformat(),
        "validationReportSha256": hashlib.sha256(report_path.read_bytes()).hexdigest(),
        "models": models,
    }
    FREEZE_OUTPUT.write_text(
        json.dumps(payload, indent=2) + "\n", encoding="utf-8"
    )
    print("Wrote", FREEZE_OUTPUT)


## Artifacts and limits

The output root contains `subset.json`, `progress.json`, `predictions/<model-id>/class-*.parquet`, `logits/<model-id>.npy`, `confusions/<model-id>.npy`, `per-class-<model-id>.csv`, `metrics.json`, `summary.json`, and `report.md`. These are local run artifacts under ignored `checkpoints/`; do not commit them.

- Validation is the tuning split. The held-out test split is a separate frozen run and must not guide model or threshold changes.
- Hint-assisted evaluation is deferred; this runner is sketch-only.
- These are recognition metrics, not game scores. A first human solve does not end a turn, and AI points are a comparison ledger.
- A high pass rate can hide a weak guesser; read coverage, accuracy among emitted guesses and overall correctness together.
- The 345-class classifier is closed-set; low confidence is not a guaranteed unknown-concept detector.


In [ ]:
REPORT_PATH = OUTPUT_ROOT / "report.md"
if REPORT_PATH.is_file():
    print(REPORT_PATH.read_text(encoding="utf-8"))
else:
    print("No report yet; run the evaluator first.")


## Stage-10 specimen examples (validation only)

This optional export selects five lowest and five highest usable stage-10 top-1 probabilities for one class from Guesser A. It reads only the completed validation prediction shard, never held-out test predictions. It recreates each selected stage-10 preprocessed image from the prepared record and verifies its SHA-256 against the evaluated input before saving a PNG. The JSONL records the image path, source row, source-file hash, validation manifest hash and bundle identity.


In [ ]:
# Export actual predictions only when the source shard and summary are present.
EXPORT_SPECIMENS = True
SPECIMEN_MODEL_ID = "guesser-vit-a-v1"
SPECIMEN_CLASS_ID = None  # None chooses the eligible class with the widest probability range
SPECIMENS_PER_BUCKET = 5
SPECIMEN_PATH = OUTPUT_ROOT / "specimens.jsonl"

if not EXPORT_SPECIMENS:
    print("Specimen export disabled.")
elif SPLIT != "validation":
    print("Specimens are restricted to validation; no export written.")
elif not SUMMARY_PATH.is_file():
    SPECIMEN_PATH.unlink(missing_ok=True)
    print("No completed summary; removed any stale specimen manifest.")
else:
    import hashlib
    import pyarrow.parquet as pq

    summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
    if SPECIMEN_MODEL_ID not in summary["models"]:
        raise ValueError(f"{SPECIMEN_MODEL_ID} is not in this completed evaluation.")
    bundle_manifest_path = BUNDLE_ROOT / SPECIMEN_MODEL_ID / "manifest.json"
    bundle_manifest = json.loads(bundle_manifest_path.read_text(encoding="utf-8"))
    shard_root = OUTPUT_ROOT / "predictions" / SPECIMEN_MODEL_ID
    shard = None
    if SPECIMEN_CLASS_ID is None:
        eligible_classes = []
        for candidate_shard in sorted(shard_root.glob("class-*.parquet")):
            sample = pq.read_table(candidate_shard, columns=["stage", "top1Prob", "preprocessStatus"]).to_pandas()
            sample = sample[(sample["stage"] == 10) & sample["top1Prob"].notna() & (sample["preprocessStatus"] == "ok")]
            if len(sample) >= 2 * SPECIMENS_PER_BUCKET:
                score_range = float(sample["top1Prob"].max() - sample["top1Prob"].min())
                class_id = int(candidate_shard.stem.removeprefix("class-"))
                eligible_classes.append((score_range, class_id, candidate_shard))
        if eligible_classes:
            # Maximize score spread; ties choose the lowest class ID for reproducibility.
            _, SPECIMEN_CLASS_ID, shard = sorted(eligible_classes, key=lambda item: (-item[0], item[1]))[0]
    else:
        shard = shard_root / f"class-{SPECIMEN_CLASS_ID:03d}.parquet"
    if shard is None or not shard.is_file():
        SPECIMEN_PATH.unlink(missing_ok=True)
        print("No class has enough usable stage-10 rows; removed any stale specimen manifest.")
    else:
        print("Selected class from actual results:", SPECIMEN_CLASS_ID)
        frame = pq.read_table(shard).to_pandas()
        frame = frame[(frame["stage"] == 10) & frame["top1Prob"].notna() & (frame["preprocessStatus"] == "ok")]
        frame = frame.sort_values(["top1Prob", "drawingId"], kind="stable")
        if len(frame) < 2 * SPECIMENS_PER_BUCKET:
            SPECIMEN_PATH.unlink(missing_ok=True)
            print(f"Only {len(frame)} usable stage-10 rows; removed any stale specimen manifest; need at least {2 * SPECIMENS_PER_BUCKET}.")
        else:
            chosen = []
            for bucket, rows in (("lowest", frame.head(SPECIMENS_PER_BUCKET)), ("highest", frame.tail(SPECIMENS_PER_BUCKET).iloc[::-1])):
                for row in rows.to_dict(orient="records"):
                    chosen.append({
                        "bucket": bucket, "split": SPLIT, "modelId": SPECIMEN_MODEL_ID,
                        "classId": int(row["classId"]), "className": row["className"],
                        "drawingId": str(row["drawingId"]), "viewIndex": int(row["viewIndex"]),
                        "stage": int(row["stage"]), "top1Label": row["top1Label"],
                        "top1Probability": float(row["top1Prob"]),
                        "top1Correct": bool(row["top1Correct"]),
                        "inputSha256": row["inputSha256"],
                        "sourcePrediction": str(shard),
                        "sourcePredictionSha256": hashlib.sha256(shard.read_bytes()).hexdigest(),
                        "manifestSha256": summary["run"]["manifestSha256"],
                        "preprocessing": summary["models"][SPECIMEN_MODEL_ID]["identity"]["preprocessing"],
                        "bundleArtifactRevision": bundle_manifest["artifactRevision"],
                        "weightsSha256": summary["models"][SPECIMEN_MODEL_ID]["identity"]["weightsSha256"],
                        "labelsSha256": summary["models"][SPECIMEN_MODEL_ID]["identity"]["labelsSha256"],
                    })
            subset = json.loads((OUTPUT_ROOT / "subset.json").read_text(encoding="utf-8"))
            class_offset = sum(len(subset["indices"][str(class_id)]) for class_id in range(SPECIMEN_CLASS_ID))
            image_root = REPOSITORY_ROOT / "presentation" / "assets" / "confidence-guesser"
            image_root.mkdir(parents=True, exist_ok=True)
            render_jobs = []
            for specimen_index, row in enumerate(chosen, start=1):
                global_original = row["viewIndex"] // 10
                local_index = global_original - class_offset
                manifest_index = subset["indices"][str(SPECIMEN_CLASS_ID)][local_index]
                image_path = (image_root / f"{row['bucket']}-{specimen_index:02d}.png").resolve()
                row["imagePath"] = str(image_path)
                render_jobs.append({"manifestIndex": manifest_index, "drawingId": row["drawingId"], "candidate": row["preprocessing"], "inputSha256": row["inputSha256"], "imagePath": str(image_path)})
            render_code = "\n".join([
                "import hashlib, json, sys",
                "import numpy as np",
                "from PIL import Image",
                "from guesser_data import IndexedDrawings",
                "from guesser_data.common import load_step",
                "prepared, split, jobs_json = sys.argv[1:]",
                "jobs = json.loads(jobs_json)",
                "with IndexedDrawings(prepared, split) as drawings:",
                "    for job in jobs:",
                "        view = list(drawings.evaluation_images(job['manifestIndex'], candidate=job['candidate']))[9]",
                "        if view['drawing_id'] != job['drawingId'] or view['status'] != 'ok': raise ValueError('Rendered specimen does not match selected prediction row')",
                "        pixels = np.ascontiguousarray(view['image'])",
                "        if hashlib.sha256(pixels.tobytes()).hexdigest() != job['inputSha256']: raise ValueError('Rendered image hash differs from evaluated stage-10 input')",
                "        settings = load_step('06-images').candidate_config(job['candidate'])",
                "        grayscale = np.clip(pixels[0] * settings['std'] + settings['mean'], 0, 1)",
                "        Image.fromarray((grayscale * 255).round().astype(np.uint8), mode='L').save(job['imagePath'])",
                "print(json.dumps({'rendered': len(jobs)}))",
            ])
            rendered = subprocess.run(["uv", "run", "--project", str(MODEL_PROJECT), "--frozen", "--no-sync", "python", "-c", render_code, str(PREPARED_DIR), SPLIT, json.dumps(render_jobs)], cwd=MODEL_PROJECT, env=MODEL_ENV, capture_output=True, text=True, check=True)
            print(rendered.stdout.strip())
            SPECIMEN_PATH.write_text("\n".join(json.dumps(row, sort_keys=True) for row in chosen) + "\n", encoding="utf-8")
            specimens = pd.DataFrame(chosen)
            print("Wrote:", SPECIMEN_PATH)
            display(specimens[["bucket", "className", "drawingId", "top1Label", "top1Probability", "top1Correct", "imagePath"]])
            from IPython.display import Image as DisplayImage, Markdown
            for specimen in chosen:
                display(Markdown(f"**{specimen['bucket'].title()} confidence — {specimen['className']} → {specimen['top1Label']} ({specimen['top1Probability']:.1%}; correct: {specimen['top1Correct']})**"))
                display(DisplayImage(filename=specimen["imagePath"]))
